# 06 — Sampling & image generation

Generate images from noise using the trained DiT and frozen VAE. Includes the with-vs-without `c_spec` ablation.

**This is the ship moment**: ALD-SC generates images.

In [ ]:
import sys
sys.path.insert(0, "../src")

import torch
import matplotlib.pyplot as plt
from ald_sc.build_prior import build_arrow_prior
from ald_sc.vae import SpectralVAE
from ald_sc.dit import MinimalDiT
from ald_sc.schedule import CosineSchedule
from ald_sc.sampling import sample_ddim, sample_ddim_steps
from ald_sc.trainer import train_vae, train_diffusion
from ald_sc.data import ToyImageDataset, build_dataloader
from ald_sc.losses import ALDSCLoss

torch.manual_seed(3407)

## 1. Train VAE + DiT (quick)

In [ ]:
F, q = 32, 8
image_size = 32
latent_size = image_size // 4

embeddings = torch.randn(64, F)
prior = build_arrow_prior(embeddings, q=q, k=4)

vae = SpectralVAE(in_channels=3, latent_channels=4, feature_dim=F, base_channels=32)
loss_fn = ALDSCLoss(prior=prior, lambda_rec=1.0, lambda_chart=0.5, lambda_smooth=0.1)
dataset = ToyImageDataset(num_samples=32, image_size=image_size, channels=3)
loader = build_dataloader(dataset, batch_size=8)

list(train_vae(loader, vae, prior, loss_fn, epochs=20, lr=1e-3))

dit = MinimalDiT(
    latent_channels=4, latent_size=latent_size, patch_size=2,
    dim=64, depth=4, num_heads=4, text_dim=0, spec_dim=3*q, cfg_dropout=0.1,
)
schedule = CosineSchedule(num_steps=1000)
list(train_diffusion(loader, vae, dit, prior, schedule, epochs=20, lr=1e-3, cfg_dropout=0.1))

vae.eval(); dit.eval()
print("Training complete.")

## 2. Generate images from noise

In [ ]:
c_spec = torch.randn(4, 3 * q)
z = sample_ddim(dit, schedule, c_spec=c_spec, batch_size=4, steps=50, seed=3407)

with torch.no_grad():
    images = vae.decode(z, c_spec, prior)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    img = images[i].permute(1, 2, 0).numpy() * 0.5 + 0.5
    axes[i].imshow(img.clip(0, 1))
    axes[i].axis("off")
plt.suptitle("Generated images with c_spec conditioning")
plt.tight_layout()
plt.savefig("../results/06_generated.png", dpi=150)
plt.show()

## 3. Ablation: with vs without c_spec

In [ ]:
c_spec_cond = torch.randn(4, 3 * q)
c_spec_uncond = torch.zeros(4, 3 * q)

z_cond = sample_ddim(dit, schedule, c_spec=c_spec_cond, batch_size=4, steps=50, seed=3407)
z_uncond = sample_ddim(dit, schedule, c_spec=c_spec_uncond, batch_size=4, steps=50, seed=3407)

with torch.no_grad():
    img_cond = vae.decode(z_cond, c_spec_cond, prior)
    img_uncond = vae.decode(z_uncond, c_spec_uncond, prior)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    img = img_cond[i].permute(1, 2, 0).numpy() * 0.5 + 0.5
    axes[0, i].imshow(img.clip(0, 1))
    axes[0, i].set_title("With c_spec")
    axes[0, i].axis("off")
    img = img_uncond[i].permute(1, 2, 0).numpy() * 0.5 + 0.5
    axes[1, i].imshow(img.clip(0, 1))
    axes[1, i].set_title("Without c_spec")
    axes[1, i].axis("off")
plt.suptitle("Ablation: with vs without spectral conditioning")
plt.tight_layout()
plt.savefig("../results/06_ablation.png", dpi=150)
plt.show()

diff = (z_cond - z_uncond).abs().mean()
print(f"Latent difference (cond vs uncond): {diff:.6f}")

## 4. Sampling trajectory visualization

In [ ]:
c_spec = torch.randn(1, 3 * q)
latents = list(sample_ddim_steps(dit, schedule, c_spec=c_spec, batch_size=1, steps=20, seed=3407))

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
indices = [0, 5, 10, 15, 19]
for j, idx in enumerate(indices):
    axes[j].imshow(latents[idx][0, 0].numpy(), cmap="viridis")
    axes[j].set_title(f"Step {idx}")
    axes[j].axis("off")
plt.suptitle("DDIM sampling trajectory (latent channel 0)")
plt.tight_layout()
plt.savefig("../results/06_trajectory.png", dpi=150)
plt.show()